In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import ttest_ind, norm
from datetime import datetime
from scipy.stats import norm
import seaborn as sns
import random

import os

### Task 1.

С одной стороны, при удалении выбросов снижается дисперсия, что приводит к увеличению чувствительности теста. С другой стороны, при удалении выбросов уменьшается размер выборки, что приводит к уменьшению чувствительности.

Сравните мощности тестов с разной долей удаляемых данных. Используйте данные о времени работы бэкенда 2022-04-01T12_df_web_logs.csv в период с 2022-03-01 по 2022-03-08. Уровень значимости — 0.05. Размеры групп — 1000 человек (размер выборок будет больше, так как на одного человека приходится много значений). Проверяем гипотезу о равенстве средних с помощью теста Стьюдента. Ожидаемый эффект — увеличение времени обработки на 1%. Эффект в синтетических А/В-тестах добавляем умножением на константу.

В ответ введите номера вариантов, упорядоченные по уменьшению мощности. Например, «12345» означает, что вариант 1 обладает наибольшей мощностью, а вариант 5 — наименьшей.

1. Удалить 0.02% выбросов;

2. Удалить 0.2% выбросов;

3. Удалить 2% выбросов;

4. Удалить 10% выбросов;

5. Удалить 20% выбросов.

Удалить 2% выбросов означает, что нужно убрать по 1% минимальных и максимальных значений выборки. То есть оставить значения, которые лежат между np.quantile(values, 0.01) и np.quantile(values, 0.99). Квантили вычислять для каждой групп отдельно.

**Решение**

In [4]:
# читаем лоокально
URL_BASE = ''

def read_database(file_name):
    return pd.read_csv(os.path.join(URL_BASE, file_name))

In [5]:
web_logs = read_database('2022-04-01T12_df_web_logs.csv')
web_logs['date'] = pd.to_datetime(web_logs['date'])
web_logs.head(3)

,user_id,page,date,load_time
0,f25239,m,2022-02-03 23:45:37,80.8
1,06d6df,m,2022-02-03 23:49:56,70.5
2,06d6df,m,2022-02-03 23:51:16,89.7


In [6]:
web_logs_hist = web_logs[
    (web_logs['date'] >= datetime(2022, 3, 1))
    & (web_logs['date'] < datetime(2022, 3, 9))
]
web_logs_hist.shape

(277116, 4)

Формируем выборки

In [7]:
users = web_logs_hist.user_id.unique()

Мощность = 1- вероятность ошибки 2-го рода (т.е. power = 1 - betta)

Вероятность ошибки 2-го рода - доля наблюдений, в которых ложно-отрицательный вывод

In [8]:
alpha = 0.05

In [49]:
for i in [0.02, 0.2, 2, 10, 20]:
    # Умножение всех значений на константу. Умножаем все значения экспериментальной группы на (1 + эффект в долях от среднего).
    p_values = []
    for _ in range(1500):
        # отбираем пользователей
        users_a, users_b = np.random.choice(users, (2, 1000,), False)
        # формируем выборки времени загрузки
        web_logs_a = web_logs_hist[web_logs_hist.user_id.isin(users_a)][['load_time']]
        # делаем выборку с синтетическим эффектом
        web_logs_b =  web_logs_hist[web_logs_hist.user_id.isin(users_b)][['load_time']] * (1 + 0.01)
        # считаем границы
        lower_a = np.quantile(web_logs_a, i/100)
        upper_a = np.quantile(web_logs_a, 1 - i/100)
        lower_b = np.quantile(web_logs_b, i/100)
        upper_b = np.quantile(web_logs_b, 1 - i/100)
        _, p_val = ttest_ind(web_logs_a[(web_logs_a.load_time >= lower_a) & (web_logs_a.load_time <= upper_a)], 
                             web_logs_b[(web_logs_b.load_time >= lower_b) & (web_logs_b.load_time <= upper_b)])
        p_values.append(p_val)
    
    errors = (np.array(p_values) > alpha).astype(int)
    part_errors = np.mean(errors)
    statistical_power = 1 - part_errors
    print(f'reduction = {i}%, part errors = {part_errors:0.4f}, power = {statistical_power:0.4f}')

reduction = 0.02%, part errors = 0.8687, power = 0.1313
reduction = 0.2%, part errors = 0.2360, power = 0.7640
reduction = 2%, part errors = 0.0480, power = 0.9520
reduction = 10%, part errors = 0.0307, power = 0.9693
reduction = 20%, part errors = 0.0160, power = 0.9840


**Ответ:**   54321

### Task 2.

Выполните то же задание, изменив способ добавления эффекта. Эффект в синтетических А/В-тестах добавляем добавлением константы к 1% данных.

В ответ введите номера вариантов упорядоченные по уменьшению мощности. Например, «12345» означает, что вариант 1 обладает наибольшей мощностью, а вариант 5 — наименьшей.

1. Удалить 0.02% выбросов;

2. Удалить 0.2% выбросов;

3. Удалить 2% выбросов;

4. Удалить 10% выбросов;

5. Удалить 20% выбросов.

Удалить 2% выбросов означает, что нужно убрать по 1% минимальных и максимальных значений выборки. То есть оставить значения, которые лежат между np.quantile(values, 0.01) и np.quantile(values, 0.99). Квантили вычислять для каждой группы отдельно.



**Решение**

In [9]:
effect = 0.01

In [13]:
for i in [0.02, 0.2, 2, 10, 20]:
    p_values = []
    for _ in range(100):
        # отбираем пользователей
        users_a, users_b = np.random.choice(users, (2, 1000,), False)
        # формируем выборки времени загрузки
        web_logs_a = web_logs_hist[web_logs_hist.user_id.isin(users_a)][['load_time']].to_numpy()
        # делаем выборку с синтетическим эффектом
        web_logs_b =  web_logs_hist[web_logs_hist.user_id.isin(users_b)][['load_time']].to_numpy()
        mean_b = np.mean(web_logs_b)
        count_b = web_logs_b.shape[0]
        # берем 1% от выборки
        target_b = int(0.01 * count_b) 
        idx = np.random.choice(count_b, target_b, replace=False)
        # добавляемая константа
        add_value = effect * mean_b * count_b / len(idx)
        mask = np.zeros(count_b)
        mask[idx] += 1
        web_logs_b = web_logs_b + mask * add_value
        # считаем границы
        lower_a = np.quantile(web_logs_a, i/100)
        upper_a = np.quantile(web_logs_a, 1 - i/100)
        lower_b = np.quantile(web_logs_b, i/100)
        upper_b = np.quantile(web_logs_b, 1 - i/100)
        _, p_val = ttest_ind(web_logs_a[(web_logs_a >= lower_a) & (web_logs_a <= upper_a)], 
                             web_logs_b[(web_logs_b >= lower_b) & (web_logs_b <= upper_b)])
        p_values.append(p_val)
    
    errors = (np.array(p_values) > alpha).astype(int)
    part_errors = np.mean(errors)
    statistical_power = 1 - part_errors
    print(f'reduction = {i}%, part errors = {part_errors:0.4f}, power = {statistical_power:0.4f}')

reduction = 0.02%, part errors = 0.7200, power = 0.2800
reduction = 0.2%, part errors = 0.0800, power = 0.9200
reduction = 2%, part errors = 0.5100, power = 0.4900
reduction = 10%, part errors = 0.5400, power = 0.4600
reduction = 20%, part errors = 0.4800, power = 0.5200


**Ответ**

Если увеличить количество экспериментов, то получится, что мощность будет расти при ограничениях 32541

### Task 3.

Реализуйте функцию process_outliers.

Шаблон решения

In [ ]:
import pandas as pd


def process_outliers(metrics, bounds, outlier_process_type):
    """Возвращает новый датафрейм с обработанными выбросами в измерениях метрики.

    :param metrics (pd.DataFrame): таблица со значениями метрики
        со столбцами ['user_id', 'metric'].
    :param bounds (tuple[float, float]): нижняя и верхняя границы метрики. Всё что
        не попало между ними считаем выбросами.
    :param outlier_process_type (str): способ обработки выбросов. Возможные варианты:
        'drop' - удаляем измерение,
        'clip' - заменяем выброс на значение ближайшей границы (lower_bound, upper_bound).
    :return df: таблица со столбцами ['user_id', 'metric']
    """
        # YOUR_CODE_HERE

Пример

In [ ]:
metrics = pd.DataFrame({'user_id': [1, 2, 3], 'metric': [1., 2, 3]})
bounds = (0.1, 2.2,)
outlier_process_type = 'drop'
result = process_outliers(metrics, bounds, outlier_process_type)
# result = pd.DataFrame({'user_id': [1, 2], 'metric': [1.0, 2.0]})

outlier_process_type = 'clip'
result = process_outliers(metrics, bounds, outlier_process_type)
# result = pd.DataFrame({'user_id': [1, 2, 3], 'metric': [1.0, 2.0, 2.2]})

**Решение**

In [21]:
import pandas as pd


def process_outliers(metrics, bounds, outlier_process_type):
    """Возвращает новый датафрейм с обработанными выбросами в измерениях метрики.

    :param metrics (pd.DataFrame): таблица со значениями метрики
        со столбцами ['user_id', 'metric'].
    :param bounds (tuple[float, float]): нижняя и верхняя границы метрики. Всё что
        не попало между ними считаем выбросами.
    :param outlier_process_type (str): способ обработки выбросов. Возможные варианты:
        'drop' - удаляем измерение,
        'clip' - заменяем выброс на значение ближайшей границы (lower_bound, upper_bound).
    :return df: таблица со столбцами ['user_id', 'metric']
    """
    if outlier_process_type == 'drop':
        return metrics[(metrics.metric >= bounds[0]) & (metrics.metric <= bounds[1])]
    else:
        metrics['metric'] = metrics['metric'].clip(lower=bounds[0], upper=bounds[1])
        return metrics

In [22]:
metrics = pd.DataFrame({'user_id': [1, 2, 3], 'metric': [1., 2, 3]})
bounds = (0.1, 2.2,)
outlier_process_type = 'drop'
result = process_outliers(metrics, bounds, outlier_process_type)
# result = pd.DataFrame({'user_id': [1, 2], 'metric': [1.0, 2.0]})

In [23]:
result

,user_id,metric
0,1,1.0
1,2,2.0


In [24]:
outlier_process_type = 'clip'
result = process_outliers(metrics, bounds, outlier_process_type)
# result = pd.DataFrame({'user_id': [1, 2, 3], 'metric': [1.0, 2.0, 2.2]})

In [25]:
result

,user_id,metric
0,1,1.0
1,2,2.0
2,3,2.2
